# AI Personalized Learning Agent
## Training Pipeline (Phases 3-5)

**Active Kernel: Python 3.13 (AI Agent)**  
All packages pre-installed. Run Cell 1 first, then run all.


## Cell 1: Setup & Package Verification


In [ ]:
import sys, subprocess, importlib, os, pickle, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# Auto-install any missing package
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'--quiet'],
                              stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)

ensure('scikit-learn','sklearn'); ensure('pandas'); ensure('numpy')
ensure('matplotlib'); ensure('seaborn')
try: import torch
except ImportError:
    print('Installing torch...')
    subprocess.check_call([sys.executable,'-m','pip','install','torch',
        '--index-url','https://download.pytorch.org/whl/cpu','--quiet'],
        stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)

import sklearn, torch

# Add project to path
PROJECT_DIR = r'd:\Day 1\Final project'
for p in [PROJECT_DIR, os.path.join(PROJECT_DIR,'src')]:
    if p not in sys.path: sys.path.insert(0, p)

print(f'Python:  {sys.executable}')
print(f'sklearn: {sklearn.__version__}')
print(f'torch:   {torch.__version__}')
print(f'pandas:  {pd.__version__}')
print(f'numpy:   {np.__version__}')
print('Setup complete!')


## Phase 3: Feature Engineering


In [ ]:
from src.feature_engineering import (
    load_processed_data, engineer_features, create_sequences_for_dkt,
    prepare_xgboost_data, prepare_recommender_data
)
df = load_processed_data()
df, feature_cols, label_encoders = engineer_features(df)
sequences = create_sequences_for_dkt(df)
X_train, X_test, y_train, y_test = prepare_xgboost_data(df, feature_cols)
interaction_df, resources = prepare_recommender_data(df)
print(f'Students: {len(df)} | Features: {len(feature_cols)} | Seqs: {len(sequences)}x{len(sequences[0])} | Interactions: {len(interaction_df)}')


In [ ]:
processed_dir = os.path.join(PROJECT_DIR,'data','processed')
df.to_csv(os.path.join(processed_dir,'features_engineered.csv'),index=False)
interaction_df.to_csv(os.path.join(processed_dir,'resource_interactions.csv'),index=False)
with open(os.path.join(processed_dir,'feature_config.pkl'),'wb') as f:
    pickle.dump({'feature_cols':feature_cols,'label_encoders':label_encoders,'resources':resources},f)
with open(os.path.join(processed_dir,'dkt_sequences.pkl'),'wb') as f:
    pickle.dump(sequences,f)
print('[SAVED] Feature data saved.')


## Phase 4a: DKT Model (LSTM)
> Input(10) → LSTM(64) → Linear(5) → Sigmoid | Target AUC > 0.75


In [ ]:
from src.knowledge_tracer.dkt_model import train_dkt, predict_mastery, save_dkt_model
models_dir = os.path.join(PROJECT_DIR,'models')
os.makedirs(models_dir,exist_ok=True)
dkt_model, dkt_auc = train_dkt(sequences, n_concepts=5, epochs=50, lr=0.001)
save_dkt_model(dkt_model, os.path.join(models_dir,'dkt_model.pth'))
print(f'DKT AUC-ROC: {dkt_auc:.4f}', 'PASSED' if dkt_auc>0.75 else 'BELOW 0.75')


## Phase 4b: Gap Detector (GradientBoosting)
> 200 trees | Predicts at-risk from 39 features


In [ ]:
from src.gap_detector.xgboost_model import train_gap_detector, get_feature_importance, save_gap_detector
import matplotlib.pyplot as plt
gap_model, gap_metrics = train_gap_detector(X_train,X_test,y_train,y_test)
feature_importance = get_feature_importance(gap_model,feature_cols)
save_gap_detector(gap_model,os.path.join(models_dir,'gap_detector.pkl'))
print(f'Accuracy={gap_metrics["accuracy"]:.3f} F1={gap_metrics["f1"]:.3f} AUC={gap_metrics["auc"]:.3f}')
names=[f[0] for f in feature_importance[:15]]; vals=[f[1] for f in feature_importance[:15]]
plt.figure(figsize=(10,5)); plt.barh(names[::-1],vals[::-1],color='#667eea',edgecolor='black')
plt.title('Top 15 Feature Importances',fontweight='bold'); plt.tight_layout(); plt.show()


## Phase 4c: Resource Recommender
> Item-item cosine similarity on 7,087 interactions


In [ ]:
from src.recommender.collab_filter import ResourceRecommender, save_recommender
recommender = ResourceRecommender()
recommender.fit(interaction_df,resources)
save_recommender(recommender,os.path.join(models_dir,'recommender.pkl'))
print('Recommender trained and saved.')


## Phase 5: AI Agent Demo
> Runs all 5 tools for 3 sample students


In [ ]:
from src.gap_detector.xgboost_model import identify_weak_areas
from src.study_planner.planner import generate_study_plan, format_study_plan
from src.progress_reporter.reporter import generate_progress_report, format_progress_report
all_mastery = predict_mastery(dkt_model,sequences,n_concepts=5)
sample_ids=[]
for pool in [df[df['at_risk']==1].index, df[(df['G3']>=10)&(df['G3']<=13)].index, df[df['G3']>=17].index]:
    if len(pool): sample_ids.append(pool[0])
reports_dir=os.path.join(PROJECT_DIR,'reports'); os.makedirs(reports_dir,exist_ok=True)
for sid in sample_ids:
    row=df.iloc[sid]; mastery=all_mastery[sid]
    feats=np.nan_to_num(df.iloc[sid][feature_cols].values.astype(float),nan=0)
    gap=identify_weak_areas(gap_model,feats,feature_cols)
    recs=recommender.recommend(sid,gap['weak_areas'],top_n=3)
    plan=generate_study_plan(sid,mastery,gap['weak_areas'],recs,row.to_dict())
    report=generate_progress_report(sid,row.to_dict(),mastery,gap['weak_areas'],gap,plan)
    rpath=os.path.join(reports_dir,f'student_{sid}_report.txt')
    with open(rpath,'w') as f: f.write(format_progress_report(report)+'\n\n'+format_study_plan(plan))
    print(f'Student #{sid}: G3={row["G3"]} | {"AT RISK" if gap["is_at_risk"] else "ON TRACK"} | {plan["intensity"].upper()}')
print('\nAll 5 tools OK! Open http://localhost:8501 after launching Streamlit.')


## Done — Launch Dashboard
```bash
cd 'd:\Day 1\Final project'
python -m streamlit run app/streamlit_app.py
```
> http://localhost:8501
